Installs required packages (LangChain, ChromaDB, PDF loader, embeddings).

Imports Colab userdata to access saved API keys.

Sets environment variables for:

LangChain tracing

LangChain API key

OpenAI API key

Ensures the notebook can use LLMs and embeddings

In [5]:
!pip install -q langchain langchain-community langchain-core
!pip install -q langchain-huggingface  # For embeddings
!pip install -q faiss-cpu  # Vector store (use faiss-gpu if you have GPU and want faster performance)
!pip install -q sentence-transformers  # For local embeddings
!pip install -q pypdf  # For loading PDF documents
!pip install -q python-dotenv  # For managing environment variables
!pip install -q langchain-groq
!pip install -q ragas  # RAG evaluation framework
!pip install -q langchain-text-splitters


Imports PyPDFLoader for reading PDF files.

Imports a text splitter to break documents into smaller chunks.

Imports Chroma vector store for embedding storage.

Imports OpenAI embeddings model.

Imports the ChatOpenAI model for generating responses.

In [28]:

# Cell 4: Load Document - Simple Machine Learning Tutorial
from langchain_community.document_loaders import PyPDFLoader

# Using TutorialsPoint ML Tutorial (simple and clear)
url = "https://www.tutorialspoint.com/machine_learning/machine_learning_tutorial.pdf"
print(f"Loading: Machine Learning Tutorial")
print(f"Source: TutorialsPoint (Beginner-Friendly)")
print(f"URL: {url}\n")

loader = PyPDFLoader(url)
documents = loader.load()


Loading: Machine Learning Tutorial
Source: TutorialsPoint (Beginner-Friendly)
URL: https://www.tutorialspoint.com/machine_learning/machine_learning_tutorial.pdf



Configures a text splitter with:

Chunk size = 1000 characters

Overlap = 200 characters

Splits the loaded PDF content into multiple smaller, overlapping chunks.

Returns the number of generated chunks.

In [31]:
# Cell 5: Split Documents into Chunks
print("TEXT SPLITTING")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Size of each chunk in characters
    chunk_overlap=200,  # Overlap between chunks to maintain context
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Split on paragraphs first, then sentences
)

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")


TEXT SPLITTING
Split into 66 chunks


Initializes the OpenAI embedding model.

Converts all text chunks into embeddings using OpenAI.

Stores embeddings inside a Chroma vector database.

Saves the vector database locally (fact_vectorstore) for reuse.

In [32]:
# Cell 6: Create Embeddings and Vector Store

# Initialize embeddings model (free and local)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store created successfully!")
print(f"Total vectors in store: {vectorstore.index.ntotal}")

# Save the vector store (optional, for reuse)
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index'")

CREATING VECTOR STORE
Vector store created successfully!
Total vectors in store: 66
✓ Vector store saved to 'faiss_index'


Reloads the previously saved Chroma vector store from disk.

Creates a retriever that returns the top 4 most relevant text chunks for any query.

In [34]:
# Cell 7: Create Retriever
print("SETTING UP RETRIEVER")

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Retrieve top 3 most similar chunks
)

print("Search type: similarity")
print("Number of chunks to retrieve (k): 3")

# Test retrieval
test_query = "What is the main topic of this document?"
retrieved_docs = retriever.invoke(test_query)

print(f"\nTest Query: {test_query}")
print(f"Retrieved {len(retrieved_docs)} documents\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Retrieved Chunk {i} ---")
    print(doc.page_content[:200])
    print()

SETTING UP RETRIEVER
Search type: similarity
Number of chunks to retrieve (k): 3

Test Query: What is the main topic of this document?
Retrieved 3 documents

--- Retrieved Chunk 1 ---
11. MACHINE LEARNING – SKILLS FOR MACHINE LEARNING ................................ ................ 26 
Necessity of Various Skills of Machine Learning ...............................................

--- Retrieved Chunk 2 ---
More advanced methods such as deep learning are hence developed to solve many 
complex problems.  
As we move ahead in this tutorial, we will  understand what Machine Learning is and how 
it is used f

--- Retrieved Chunk 3 ---
Practicing one topic at a time, very soon you would acquire the width that is eventually 
required of a Machine Learning expert.  
Good Luck!  
13. Machine Learning – Conclusion



This cell builds a complete RAG (Retrieval-Augmented Generation) pipeline. It starts by defining a prompt template that tells the LLM to answer questions only using retrieved context. A helper function formats the retrieved documents into a single text block. Then a RAG chain is created: the retriever fetches relevant chunks, the question is passed through unchanged, the prompt is filled with both question and context, the LLM generates an answer, and the output is parsed into a clean string. Finally, it prints a message confirming the chain was created successfully.

In [36]:
# Cell 8: Create RAG Chain
# Define the prompt template
template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer based on the context, just say that you don't know.
Keep the answer concise and accurate.

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Format documents function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully!")

RAG chain created successfully!


This cell runs multiple example questions through the RAG system. It loops over a list of predefined machine-learning questions, prints each one, and then retrieves the most relevant text chunks using the retriever. It displays the retrieved context so you can see what information the model is using. After that, it sends the question into the RAG chain, which combines the context and prompt to generate an answer. Finally, it prints the model’s response for each question.

In [38]:
# Cell 9: Ask Questions and Get Answers

# Example questions
questions = [
   "What is machine learning and how is it defined?",
    "What are the main types or categories of machine learning?",
    "Can you explain supervised learning with an example?",
    "What is the difference between classification and regression?",
    "What are some real-world applications of machine learning?"
]

for i, question in enumerate(questions, 1):
    print(f"Question {i}: {question}")

    # Get retrieved context
    retrieved = retriever.invoke(question)
    print("\n--- Retrieved Context ---")
    for j, doc in enumerate(retrieved, 1):
        print(f"\nChunk {j}:")
        print(doc.page_content[:300] + "...")

    # Get answer
    answer = rag_chain.invoke(question)
    print(f"\n--- Generated Answer ---")
    print(answer)
    print()

Question 1: What is AI and how is it defined?

--- Retrieved Context ---

Chunk 1:
Supervised Learning is where the AI really began its journey. This technique was applied 
successfully in several cases. You have used this model while doing the hand -written 
recognition on your machine. Several algorithms have been developed for supervised 
learning. You will learn about them in ...

Chunk 2:
More advanced methods such as deep learning are hence developed to solve many 
complex problems.  
As we move ahead in this tutorial, we will  understand what Machine Learning is and how 
it is used for developing such complex AI applications.  
  
3. Machine Learning – Traditional AI...

Chunk 3:
that we have today are much more complex and use techniques far beyond the statistical 
techniques used by the early AI programs.  
Some of the examples of statistical techniques that are used for developing AI applications 
in those days and are still in practice are listed here: 
 
 Regression 
...


This cell performs a simple evaluation of the RAG system. It defines test questions along with expected keywords that should appear in a correct answer. For each test case, it asks the question using the RAG chain, prints the generated answer, and then checks how many of the expected keywords appear in the response. It reports which keywords were found and calculates a basic relevance score, giving a quick way to gauge whether the model is responding accurately based on the document.

In [41]:
# Cell 11: Simple Evaluation
# Create test questions with expected answers
test_cases = [
    {
        "question": "What is the main topic discussed in the document?",
        "expected_keywords": ["Machine Learning", "Supervised Learning", "AI"]  # Adjust based on your document
    },
    # Add more test cases
]


for i, test in enumerate(test_cases, 1):
    print(f"Test Case {i}:")
    print(f"Question: {test['question']}")

    # Get answer
    answer = rag_chain.invoke(test['question'])
    print(f"Answer: {answer}")

    # Check if expected keywords are present
    keywords_found = [kw for kw in test['expected_keywords']
                      if kw.lower() in answer.lower()]

    print(f"Expected keywords: {test['expected_keywords']}")
    print(f"Keywords found: {keywords_found}")
    print(f"Relevance score: {len(keywords_found)}/{len(test['expected_keywords'])}")
    print()


Test Case 1:
Question: What is the main topic discussed in the document?
Answer: The main topic discussed in the document is Machine Learning.
Expected keywords: ['Machine Learning', 'Supervised Learning', 'AI']
Keywords found: ['Machine Learning', 'AI']
Relevance score: 2/3



This notebook builds a complete Retrieval-Augmented Generation (RAG) system using LangChain, OpenAI embeddings, and a Chroma vector database. The workflow begins by installing and configuring all necessary libraries, including setting API keys for LangChain tracing and OpenAI. A PDF document is then loaded, and its content is split into manageable text chunks to prepare it for embedding. These chunks are converted into vector representations using OpenAI’s embedding model and stored inside a persistent Chroma vector store, making retrieval efficient and repeatable.

After saving and reloading the vector store, a retriever is created to fetch the most relevant chunks for any user query. A chat model (GPT-4o-mini) is initialized, and a custom RAG chain is constructed. This chain retrieves context, formats it, injects it into a prompt template, and generates an answer that stays grounded in the document. The notebook then demonstrates the system by running multiple example questions, showing both the retrieved context and the generated answers. Finally, a simple evaluation loop checks whether the model’s responses contain expected keywords, providing a basic measure of relevance and accuracy.

Overall, the notebook guides the user from document ingestion, through embedding and retrieval, to generation and evaluation—resulting in a fully functional RAG question-answering pipeline.